# Playground · K-Means

**Tópicos de Inteligencia de Negocios** · Clustering NO supervisado

> K-Means **descubre grupos** en datos sin etiquetas. Tú le dices cuántos quieres (K), él coloca centroides al azar, asigna puntos al centroide más cercano, recalcula, y repite. Acá podrás ver el proceso y entender por qué el método del codo y la silueta son útiles.

¿Qué vas a poder hacer aquí?

1. Generar datasets 2D sin etiquetas (blobs, lunas, anisotrópicos).
2. Variar **K**, **inicialización** (k-means++ vs random) y **n_init**.
3. Ver el **método del codo** y el **coeficiente de silueta** para elegir K.
4. Subir tu propio CSV y descubrir grupos.

> 🔍 **Tip:** los botones **`?`** explican cada parámetro. Configura → presiona **🚀 Entrenar modelo**.


## Marco Teórico

### El algoritmo (3 minutos)
1. **Elige K** (decisión humana — el reto principal).
2. **Inicializa K centroides** (al azar o con k-means++).
3. **Asignación:** cada punto va al centroide más cercano (euclidiana).
4. **Actualización:** cada centroide se mueve al promedio de sus puntos asignados.
5. **Repite** 3 y 4 hasta que nada se mueva (o llegues a `max_iter`).

### Función objetivo: Inercia (within-cluster sum of squares)
$$\text{Inercia} = \sum_{i=1}^{m} \min_{c_k} \| x^{(i)} - c_k \|^2$$

K-Means **minimiza** esta inercia. Es decir, busca centroides que dejen a cada punto lo más cerca posible de su centroide asignado.

> ⚠️ **Cuidado:** la inercia siempre baja al subir K (en el límite, K = n da inercia 0). Por eso necesitas otros criterios para elegir K.

### Cómo elegir K

| Método | Cómo funciona |
|--------|---------------|
| **Método del codo** | Grafica inercia vs K. Busca el "codo" donde la mejora se desacelera. |
| **Silueta** | Mide qué tan dentro de su cluster están los puntos. Rango [-1, 1]; > 0.5 es bueno. |
| **Conocimiento del dominio** | A veces sabes que hay 3 segmentos de cliente, K=3. Punto. |

### Limitaciones importantes

- Asume **clusters esféricos** y de tamaños similares — falla con lunas, anillos, anisotrópicos.
- Es **sensible a la inicialización** — por eso `n_init > 1` corre varias veces y se queda con la mejor.
- Las **distancias importan** — siempre normaliza features con escalas distintas.
- Requiere **K predefinido** — no puede decidir solo cuántos clusters hay.


## 1. Configuración inicial

In [ ]:
# ===== Auto-instalar paquetes que no vienen precargados en Pyodide (JupyterLite) =====
# Pyodide ya trae numpy, pandas, matplotlib y scikit-learn,
# pero ipywidgets hay que instalarlo en caliente la primera vez (~10 segundos).
try:
    import ipywidgets  # noqa: F401
except ImportError:
    import micropip  # disponible solo en Pyodide
    print('⏳ Instalando ipywidgets en el navegador (1 vez)...')
    await micropip.install('ipywidgets')
    print('✅ ipywidgets instalado')

import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import silhouette_score

import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['figure.dpi'] = 90
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('✅ Listo. Librerías cargadas.')


## 2. Funciones auxiliares

In [ ]:
def con_ayuda(control, explicacion):
    btn = widgets.Button(description='?', button_style='info', tooltip=explicacion,
                         layout=widgets.Layout(width='30px', height='28px', margin='0 0 0 4px'))
    panel = widgets.HTML(
        value=(f'<div style="background:#dbeafe; padding:8px 10px; border-radius:4px; '
               f'border-left:3px solid #2563eb; margin:2px 0 8px 18px; font-size:12px; '
               f'color:#1e3a8a;">💡 {explicacion}</div>'),
        layout=widgets.Layout(display='none'))
    btn.on_click(lambda _: setattr(panel.layout, 'display',
                                   'none' if panel.layout.display != 'none' else 'block'))
    return widgets.VBox([widgets.HBox([control, btn]), panel])


def generar_dataset_clustering(tipo='Blobs', n=300, ruido=0.5, n_grupos=3, seed=42):
    """Datasets sin etiquetas (descartamos las y reales)."""
    if tipo == 'Blobs (esféricos)':
        X, _ = make_blobs(n_samples=n, centers=n_grupos, cluster_std=ruido*1.5+0.3,
                          random_state=seed)
    elif tipo == 'Blobs anisotrópicos':
        X, _ = make_blobs(n_samples=n, centers=n_grupos, cluster_std=ruido*1.5+0.3,
                          random_state=seed)
        transform = np.array([[0.6, -0.6], [-0.4, 0.8]])
        X = X @ transform
    elif tipo == 'Lunas (K-Means falla)':
        X, _ = make_moons(n_samples=n, noise=ruido*0.3, random_state=seed)
    elif tipo == 'Tamaños distintos':
        sizes = [int(n*0.6), int(n*0.3), int(n*0.1)][:n_grupos]
        sizes += [int(n*0.1)] * (n_grupos - len(sizes))
        X, _ = make_blobs(n_samples=sum(sizes[:n_grupos]), centers=n_grupos,
                          cluster_std=ruido*1.5+0.3, random_state=seed)
    return X


def construir_kmeans(k, init, n_init, max_iter):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('modelo', KMeans(n_clusters=k, init=init, n_init=n_init,
                          max_iter=max_iter, random_state=42)),
    ])


def graficar_resultados_km(X, pipeline, k, titulo=''):
    fig = plt.figure(figsize=(14, 5))
    gs = fig.add_gridspec(1, 3, wspace=0.3)
    ax1 = fig.add_subplot(gs[0, :2])
    ax2 = fig.add_subplot(gs[0, 2])

    modelo = pipeline.named_steps['modelo']
    scaler = pipeline.named_steps['scaler']
    labels = modelo.labels_
    # Centroides en espacio original (deshacer el scaling)
    centros_escalados = modelo.cluster_centers_
    centros = scaler.inverse_transform(centros_escalados)

    cmap = plt.cm.tab10
    for i in range(k):
        mask = labels == i
        ax1.scatter(X[mask,0], X[mask,1], c=[cmap(i)], s=40,
                    edgecolor='white', linewidth=0.5, alpha=0.85,
                    label=f'Cluster {i} (n={mask.sum()})')
    ax1.scatter(centros[:,0], centros[:,1], c='black', marker='X', s=300,
                edgecolor='yellow', linewidth=2.5, label='Centroides', zorder=10)
    ax1.set_title(f'Clusters descubiertos {titulo}', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Feature 1'); ax1.set_ylabel('Feature 2')
    ax1.legend(loc='best', fontsize=9, framealpha=0.9)

    # Método del codo + silueta (paralelo)
    inercias = []; siluetas = []
    Ks = list(range(2, min(11, len(X)//2)))
    X_scaled = scaler.transform(X)
    for kk in Ks:
        km = KMeans(n_clusters=kk, n_init=5, random_state=42)
        lbl = km.fit_predict(X_scaled)
        inercias.append(km.inertia_)
        try:
            siluetas.append(silhouette_score(X_scaled, lbl))
        except Exception:
            siluetas.append(0)

    ax2_b = ax2.twinx()
    l1 = ax2.plot(Ks, inercias, 'o-', color='#2563eb', linewidth=2, label='Inercia')
    l2 = ax2_b.plot(Ks, siluetas, 's-', color='#dc2626', linewidth=2, label='Silueta')
    ax2.axvline(x=k, color='#16a34a', linestyle='--', alpha=0.7,
                label=f'K elegido = {k}')
    ax2.set_xlabel('K (número de clusters)')
    ax2.set_ylabel('Inercia (suma de distancias²)', color='#2563eb')
    ax2_b.set_ylabel('Coef. silueta', color='#dc2626')
    ax2.set_title('Codo + Silueta', fontsize=11, fontweight='bold')
    lns = l1 + l2 + [ax2.lines[-1]]
    ax2.legend(handles=lns, loc='center right', fontsize=9)
    ax2_b.set_ylim(-0.2, 1.0)

    plt.tight_layout(); plt.show()

    # Métricas en texto
    sil_actual = silhouette_score(X_scaled, labels) if k > 1 else float('nan')
    print(f'📊 Métricas del clustering con K={k}:')
    print(f'  Inercia:           {modelo.inertia_:.2f}')
    print(f'  Coef. silueta:     {sil_actual:.4f}    (>0.5 bueno, >0.7 excelente)')
    print(f'  Iteraciones:       {modelo.n_iter_}')
    print(f'  Tamaño de cada cluster:')
    for i in range(k):
        pct = 100 * (labels == i).sum() / len(labels)
        bar = '█' * int(pct / 2.5)
        print(f'    Cluster {i}: {(labels == i).sum():4d} pts ({pct:5.1f}%)  {bar}')


## 3. Playground · Datos sintéticos 🧪

> 🔍 Botones **`?`** para ayuda. Configura → **🚀 Entrenar modelo**.


In [ ]:
# DATOS
w_tipo = widgets.Dropdown(
    options=['Blobs (esféricos)', 'Blobs anisotrópicos', 'Lunas (K-Means falla)', 'Tamaños distintos'],
    value='Blobs (esféricos)', description='Dataset:',
    style={'description_width':'130px'})
w_n = widgets.IntSlider(value=300, min=50, max=800, step=20,
                        description='n puntos:', style={'description_width':'130px'})
w_ruido = widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1,
                              description='Dispersión:', style={'description_width':'130px'})
w_grupos = widgets.IntSlider(value=3, min=2, max=6,
                             description='Grupos REALES:', style={'description_width':'130px'})
w_seed = widgets.IntSlider(value=42, min=0, max=100,
                           description='Semilla:', style={'description_width':'130px'})

# MODELO
w_k = widgets.IntSlider(value=3, min=2, max=10, step=1,
                        description='K (clusters):', style={'description_width':'130px'})
w_init = widgets.Dropdown(options=['k-means++', 'random'], value='k-means++',
                          description='Inicialización:', style={'description_width':'130px'})
w_ninit = widgets.IntSlider(value=10, min=1, max=30,
                            description='n_init:', style={'description_width':'130px'})
w_maxiter = widgets.IntSlider(value=300, min=10, max=500, step=10,
                              description='max_iter:', style={'description_width':'130px'})

# Explicaciones
ay_tipo = ('Forma del dataset. Blobs esféricos son lo ideal para K-Means. '
           'Lunas, anisotrópicos y tamaños distintos son trampas comunes — '
           '¡pruébalos para ver dónde K-Means falla!')
ay_n = 'Cantidad de puntos.'
ay_ruido = 'Qué tan dispersos están los grupos. Disp. baja = clusters bien definidos.'
ay_grupos = ('Cantidad VERDADERA de grupos en los datos (separado de K que tú eliges en el modelo). '
             '¡Compara variarlo vs variar K!')
ay_seed = 'Misma semilla = mismos datos.'
ay_k = ('Número de clusters a buscar. ESTE es el parámetro estrella de K-Means. '
        'Compara K = grupos reales vs K mayor/menor.')
ay_init = ('Cómo inicializar centroides. k-means++ es el default — más inteligente '
           '(elige puntos lejanos entre sí). random es la versión clásica.')
ay_ninit = ('Cuántas veces se corre K-Means con distintas inicializaciones. '
            'Se queda con el resultado de menor inercia. Sube esto para resultados '
            'más estables (a costa de tiempo).')
ay_maxiter = 'Máximo de iteraciones por corrida. 300 casi siempre es suficiente.'

panel_d = widgets.VBox([
    widgets.HTML('<b>📊 Datos</b>'),
    con_ayuda(w_tipo, ay_tipo), con_ayuda(w_n, ay_n),
    con_ayuda(w_ruido, ay_ruido), con_ayuda(w_grupos, ay_grupos),
    con_ayuda(w_seed, ay_seed),
])
panel_m = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>'),
    con_ayuda(w_k, ay_k), con_ayuda(w_init, ay_init),
    con_ayuda(w_ninit, ay_ninit), con_ayuda(w_maxiter, ay_maxiter),
])
controles = widgets.HBox([panel_d, panel_m])
salida = widgets.Output()

def actualizar():
    with salida:
        clear_output(wait=True)
        try:
            X = generar_dataset_clustering(w_tipo.value, w_n.value, w_ruido.value,
                                            w_grupos.value, w_seed.value)
            pipe = construir_kmeans(w_k.value, w_init.value, w_ninit.value, w_maxiter.value)
            pipe.fit(X)
            graficar_resultados_km(X, pipe, w_k.value,
                                    titulo=f'· K={w_k.value}, {w_init.value}')
        except Exception as e:
            print(f'⚠️ {e}')

btn_e = widgets.Button(description='🚀 Entrenar modelo', button_style='primary',
                       layout=widgets.Layout(width='220px', height='40px',
                                             margin='10px 0 6px 0'))
estado = widgets.HTML(value='<span style="color:#64748b;font-style:italic;">'
                            'Configura los parámetros y haz clic en <b>Entrenar modelo</b>.</span>')

def _stale(*_):
    estado.value = ('<span style="color:#ea580c;">🔄 <b>Cambios sin aplicar.</b> '
                    'Haz clic en <b>Entrenar modelo</b>.</span>')

def _go(_):
    btn_e.disabled = True; btn_e.description = '⏳ Entrenando...'
    estado.value = '<span style="color:#2563eb;">⏳ Entrenando...</span>'
    try:
        actualizar()
        estado.value = ('<span style="color:#16a34a;">✅ <b>Modelo entrenado.</b> '
                        'Cambia parámetros y vuelve a entrenar.</span>')
    except Exception as e:
        estado.value = f'<span style="color:#dc2626;">❌ {e}</span>'
    finally:
        btn_e.disabled = False; btn_e.description = '🚀 Entrenar modelo'

btn_e.on_click(_go)
for w in [w_tipo, w_n, w_ruido, w_grupos, w_seed, w_k, w_init, w_ninit, w_maxiter]:
    w.observe(_stale, names='value')

display(controles, btn_e, estado, salida)
actualizar()
estado.value = '<span style="color:#16a34a;">✅ <b>Modelo entrenado con la configuración inicial.</b></span>'


## 4. Playground · Sube tu CSV 📂

Sube un CSV con al menos 2 columnas numéricas. K-Means **no necesita etiquetas** — solo features.


In [ ]:
estado_csv = {'df': None}

w_upload = widgets.FileUpload(accept='.csv', multiple=False, description='📁 Subir CSV')
w_x1 = widgets.Dropdown(options=[], description='Feature 1:', style={'description_width':'130px'})
w_x2 = widgets.Dropdown(options=[], description='Feature 2:', style={'description_width':'130px'})

w_k2 = widgets.IntSlider(value=3, min=2, max=10,
                         description='K (clusters):', style={'description_width':'130px'})
w_init2 = widgets.Dropdown(options=['k-means++','random'], value='k-means++',
                           description='Inicialización:', style={'description_width':'130px'})
w_ninit2 = widgets.IntSlider(value=10, min=1, max=30,
                             description='n_init:', style={'description_width':'130px'})

salida_csv = widgets.Output()
salida_info = widgets.Output()

def on_upload(change):
    with salida_info:
        clear_output(wait=True)
        if not w_upload.value: return
        try:
            archivo = w_upload.value[0] if isinstance(w_upload.value, tuple) else next(iter(w_upload.value.values()))
            contenido = archivo['content']
            nombre = archivo.get('name','archivo.csv')
        except Exception:
            archivo = list(w_upload.value.values())[0]
            contenido = archivo['content']
            nombre = archivo.get('metadata',{}).get('name','archivo.csv')
        try:
            df = pd.read_csv(io.BytesIO(bytes(contenido)))
        except Exception as e:
            print(f'⚠️ {e}'); return
        estado_csv['df'] = df
        cn = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(cn) < 2:
            print(f'⚠️ Necesitas ≥ 2 columnas numéricas. Encontré: {cn}'); return
        w_x1.options = cn; w_x2.options = cn
        w_x1.value = cn[0]; w_x2.value = cn[1]
        print(f'✅ {nombre} — {df.shape[0]}×{df.shape[1]}')
        display(df.head())

w_upload.observe(on_upload, names='value')

def actualizar2():
    with salida_csv:
        clear_output(wait=True)
        df = estado_csv['df']
        if df is None: print('⬆️ Sube un CSV primero.'); return
        try:
            sub = df[[w_x1.value, w_x2.value]].dropna()
            X = sub.values
            pipe = construir_kmeans(w_k2.value, w_init2.value, w_ninit2.value, 300)
            pipe.fit(X)
            graficar_resultados_km(X, pipe, w_k2.value,
                                    titulo=f'· {w_x2.value} vs {w_x1.value}')
        except Exception as e:
            print(f'⚠️ {e}')

btn_e2 = widgets.Button(description='🚀 Entrenar modelo', button_style='primary',
                        layout=widgets.Layout(width='220px', height='40px',
                                              margin='10px 0 6px 0'))
estado2 = widgets.HTML(value='<span style="color:#64748b;font-style:italic;">'
                             'Sube un CSV y haz clic en <b>Entrenar modelo</b>.</span>')

def _stale2(*_):
    if estado_csv['df'] is None: return
    estado2.value = '<span style="color:#ea580c;">🔄 <b>Cambios sin aplicar.</b></span>'

def _go2(_):
    if estado_csv['df'] is None:
        estado2.value = '<span style="color:#dc2626;">⚠️ Primero sube un CSV.</span>'; return
    btn_e2.disabled = True; btn_e2.description = '⏳ Entrenando...'
    estado2.value = '<span style="color:#2563eb;">⏳ Entrenando...</span>'
    try:
        actualizar2()
        estado2.value = '<span style="color:#16a34a;">✅ <b>Modelo entrenado.</b></span>'
    except Exception as e:
        estado2.value = f'<span style="color:#dc2626;">❌ {e}</span>'
    finally:
        btn_e2.disabled = False; btn_e2.description = '🚀 Entrenar modelo'

btn_e2.on_click(_go2)
for w in [w_x1, w_x2, w_k2, w_init2, w_ninit2]:
    w.observe(_stale2, names='value')

panel_c = widgets.VBox([
    widgets.HTML('<b>📁 Datos (CSV)</b>'),
    con_ayuda(w_upload, 'Sube tu .csv (≥ 2 columnas numéricas, sin etiqueta).'),
    con_ayuda(w_x1, 'Primera feature para clustering.'),
    con_ayuda(w_x2, 'Segunda feature para clustering.'),
])
panel_m2 = widgets.VBox([
    widgets.HTML('<b>🧮 Modelo</b>'),
    con_ayuda(w_k2, ay_k), con_ayuda(w_init2, ay_init),
    con_ayuda(w_ninit2, ay_ninit),
])
display(widgets.HBox([panel_c, panel_m2]), btn_e2, estado2, salida_info, salida_csv)


## 5. Ejercicios guiados 📝

### Ejercicio 1 — Encontrar el K real
1. Dataset = **Blobs (esféricos)**, n = 300, dispersión = 0.5, grupos reales = 4.
2. Prueba K = 2, 3, 4, 5, 6.
3. **Pregunta:** ¿En qué K hace "codo" la curva de inercia (panel derecho)? ¿Coincide con la silueta máxima? ¿Acierta al número real (4)?

### Ejercicio 2 — Cuando K-Means falla
1. Dataset = **Lunas**. K = 2.
2. **Pregunta:** ¿Los clusters tienen sentido? ¿Por qué K-Means corta las lunas a la mitad?

### Ejercicio 3 — Sensibilidad a la inicialización
1. Dataset = **Blobs (esféricos)** con grupos = 5, dispersión alta = 1.2.
2. Prueba `init = random` con `n_init = 1`. Cambia la semilla varias veces (42, 7, 99).
3. Ahora cambia a `init = k-means++` con `n_init = 10`. Cambia la semilla.
4. **Pregunta:** ¿Cuál combo da resultados más estables?

### Ejercicio 4 — Anisotrópicos
1. Dataset = **Blobs anisotrópicos**, grupos = 3, K = 3.
2. **Pregunta:** ¿K-Means encuentra los clusters "reales" o se confunde? ¿Qué dice esto sobre las asunciones del algoritmo?

### Ejercicio 5 — Tu propio CSV
Sube un dataset (clientes de un retail, países por indicadores, etc.). Aplica K-Means con K=3, 4, 5. Justifica con el método del codo cuál K elegir.


## 6. Resumen

- K-Means es **rápido y simple**, ideal cuando los grupos son esféricos y de tamaño similar.
- **K es tu decisión** — usa el codo, la silueta, y/o conocimiento del dominio.
- Es **sensible a la inicialización** — siempre usa `n_init ≥ 10` y `k-means++`.
- **Normaliza tus features** antes de aplicarlo (este playground ya lo hace internamente).
- Para clusters no esféricos, considera DBSCAN, clustering jerárquico, o Gaussian Mixture Models.

---

> *Tópicos de Inteligencia de Negocios · Playground de Machine Learning*
